In [1]:
import sys, torch, numpy
print("Python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("CUDA   :", torch.version.cuda, "| available:", torch.cuda.is_available())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("numpy  :", numpy.__version__)

Python : 3.12.13
torch  : 2.10.0+cu128
CUDA   : 12.8 | available: True
GPU    : Tesla T4
numpy  : 2.4.6


In [2]:
%cd /kaggle/working
!git clone --branch 0.3.0 --depth 1 https://github.com/Megvii-BaseDetection/YOLOX.git
%cd /kaggle/working/YOLOX
!git rev-parse HEAD   # record this commit hash for the paper's methods section

# Install YOLOX's runtime deps explicitly, then install YOLOX itself with --no-deps
# so its pinned requirements.txt can't downgrade torch/numpy and break Kaggle's CUDA build.
!pip install -q loguru thop ninja tabulate pycocotools
!pip install -q -e . --no-deps


/kaggle/working
Cloning into 'YOLOX'...
remote: Enumerating objects: 216, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 216 (delta 22), reused 109 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (216/216), 2.80 MiB | 17.39 MiB/s, done.
Resolving deltas: 100% (22/22), done.
Note: switching to '419778480ab6ec0590e5d3831b3afb3b46ab2aa3'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

/kaggle/working/YOLOX
419778480ab6ec0590e5d3831b3afb3b46ab2aa3

In [3]:
import torch   # (the Windows torch-before-albumentations DLL rule is Windows-only; harmless habit here)

# COCO-pretrained yolox_s (~40.5 mAP) — warm-start for the fine-tune.
# -O forces an ABSOLUTE output path, so the file lands where every later cell expects
# it, regardless of the current working directory.
!wget -q -O /kaggle/working/yolox_s.pth https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth

# (a) Tight check: does the checkpoint match the yolox-s architecture?
from yolox.exp import get_exp
exp = get_exp(None, "yolox-s")
model = exp.get_model()
ckpt = torch.load("/kaggle/working/yolox_s.pth", map_location="cpu", weights_only=False)  # trusted official file
model.load_state_dict(ckpt["model"])
model.eval().cuda()
print("Model + weights OK — params:", round(sum(p.numel() for p in model.parameters()) / 1e6, 2), "M")

# (b) End-to-end check: full inference pipeline on the bundled demo image.
!python tools/demo.py image -n yolox-s -c /kaggle/working/yolox_s.pth \
    --path assets/dog.jpg --conf 0.25 --nms 0.45 --tsize 640 --save_result --device gpu


Model + weights OK — params: 8.97 M
2026-06-02 12:20:52.814 | INFO     | __main__:main:259 - Args: Namespace(demo='image', experiment_name='yolox_s', name='yolox-s', path='assets/dog.jpg', camid=0, save_result=True, exp_file=None, ckpt='/kaggle/working/yolox_s.pth', device='gpu', conf=0.25, nms=0.45, tsize=640, fp16=False, legacy=False, fuse=False, trt=False)
2026-06-02 12:20:53.368 | INFO     | __main__:main:269 - Model Summary: Params: 8.97M, Gflops: 26.93
2026-06-02 12:20:53.675 | INFO     | __main__:main:282 - loading checkpoint
2026-06-02 12:20:53.808 | INFO     | __main__:main:286 - loaded checkpoint done.
2026-06-02 12:20:55.066 | INFO     | __main__:inference:165 - Infer time: 1.2017s
2026-06-02 12:20:55.074 | INFO     | __main__:image_demo:202 - Saving detection result in ./YOLOX_outputs/yolox_s/vis_res/2026_06_02_12_20_53/dog.jpg


In [4]:
import os
from pathlib import Path

SRC = Path("/kaggle/input/datasets/lazarvelinov46/anti-uav-v4-track3-yolo")
DST = Path("/kaggle/working/anti_uav_v4")

if DST.exists() or DST.is_symlink():
    print(f"{DST} already present.")
else:
    print(f"Top-level contents of {SRC}:")
    for p in sorted(SRC.iterdir()):
        print(f"  {'DIR ' if p.is_dir() else 'FILE'}  {p.name}")
    print()

    data_root = next((c for c in (SRC / "anti_uav_v4", SRC)
                      if (c / "images" / "train").is_dir()
                      and (c / "labels" / "train").is_dir()), None)
    if data_root is None:
        raise RuntimeError(f"Could not find images/train + labels/train under {SRC}")

    os.symlink(data_root, DST, target_is_directory=True)
    print(f"Symlinked {DST} -> {data_root}\n")

for split in ("train", "val"):
    n_img = len(list((DST / "images" / split).glob("*.jpg")))
    n_lbl = len(list((DST / "labels" / split).glob("*.txt")))
    print(f"  {split:<5}: {n_img:>7} images, {n_lbl:>7} labels  (match: {n_img == n_lbl})")


Top-level contents of /kaggle/input/datasets/lazarvelinov46/anti-uav-v4-track3-yolo:
  DIR   anti_uav_v4

Symlinked /kaggle/working/anti_uav_v4 -> /kaggle/input/datasets/lazarvelinov46/anti-uav-v4-track3-yolo/anti_uav_v4

  train:  122488 images,  122488 labels  (match: True)
  val  :   30093 images,   30093 labels  (match: True)


In [5]:
import os, json
from pathlib import Path
from PIL import Image
try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **k: x

YOLO_ROOT  = Path("/kaggle/working/anti_uav_v4")     # images/{train,val}, labels/{train,val}
YOLOX_ROOT = Path("/kaggle/working/yolox_data")      # writable root we control
ANN_DIR    = YOLOX_ROOT / "annotations"
ANN_DIR.mkdir(parents=True, exist_ok=True)

IMG_W, IMG_H = 640, 512                               # Anti-UAV v4 thermal IR; checked below
CATEGORIES   = [{"id": 1, "name": "uav", "supercategory": "none"}]
SPLITS       = {"train": "instances_train.json", "val": "instances_val.json"}

def verify_dims(img_dir):
    first = next(iter(sorted(img_dir.glob("*.jpg"))), None)
    if first is None:
        raise RuntimeError(f"No .jpg in {img_dir}")
    w, h = Image.open(first).size
    if (w, h) != (IMG_W, IMG_H):
        print(f"  [WARN] {img_dir.name}: image is {w}x{h}, expected {IMG_W}x{IMG_H}")
    return w, h

def yolo_to_coco(split, json_name):
    img_dir, lbl_dir = YOLO_ROOT / "images" / split, YOLO_ROOT / "labels" / split

    # Symlink the read-only image dir into the writable root under YOLOX's expected
    # default folder name ("train2017"/"val2017") — this YOLOX builds COCODataset
    # inline with those names, so the folders must match.
    link = YOLOX_ROOT / f"{split}2017"
    if not link.exists() and not link.is_symlink():
        os.symlink(img_dir.resolve(), link, target_is_directory=True)

    out_path = ANN_DIR / json_name
    if out_path.exists():
        print(f"  {json_name} exists — skipping (delete to regenerate).")
        return

    w, h = verify_dims(img_dir)
    images, annotations, ann_id = [], [], 1

    for img_id, img_path in enumerate(tqdm(sorted(img_dir.glob("*.jpg")), desc=f"  {split}"), start=1):
        images.append({"id": img_id, "file_name": img_path.name, "width": w, "height": h})

        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue                                  # background frame: kept, zero boxes

        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            _cls, cx, cy, bw, bh = map(float, parts)  # class always 0 (uav)
            # YOLO normalized cx,cy,w,h -> COCO absolute-pixel x,y,w,h (top-left)
            x, y = (cx - bw / 2) * w, (cy - bh / 2) * h
            aw, ah = bw * w, bh * h
            annotations.append({
                "id": ann_id, "image_id": img_id,
                "category_id": 1,                     # 1-indexed; YOLOX maps it to class 0
                "bbox": [round(x, 2), round(y, 2), round(aw, 2), round(ah, 2)],
                "area": round(aw * ah, 2), "iscrowd": 0, "segmentation": [],
            })
            ann_id += 1

    with open(out_path, "w") as f:
        json.dump({"images": images, "annotations": annotations, "categories": CATEGORIES}, f)
    print(f"  Wrote {out_path.name}: {len(images)} images, {len(annotations)} annotations")

print("Converting YOLO -> COCO ...")
for split, json_name in SPLITS.items():
    yolo_to_coco(split, json_name)
print("Done.")


Converting YOLO -> COCO ...


  train: 100%|██████████| 122488/122488 [13:31<00:00, 150.99it/s]


  Wrote instances_train.json: 122488 images, 2564104 annotations


  val: 100%|██████████| 30093/30093 [02:58<00:00, 168.65it/s]


  Wrote instances_val.json: 30093 images, 604854 annotations
Done.


In [6]:
from pycocotools.coco import COCO
import numpy as np

for split, json_name in SPLITS.items():
    coco = COCO(str(ANN_DIR / json_name))             # raises if the JSON is malformed
    img_ids, ann_ids = coco.getImgIds(), coco.getAnnIds()
    cats = coco.loadCats(coco.getCatIds())
    areas = [a["area"] for a in coco.loadAnns(ann_ids)]
    n_bg = sum(1 for i in img_ids if not coco.getAnnIds(imgIds=i))

    print(f"\n[{split}]  {json_name}")
    print(f"  categories  : {[c['name'] for c in cats]} (id {[c['id'] for c in cats]})")
    print(f"  images      : {len(img_ids)}  (background-only: {n_bg})")
    print(f"  annotations : {len(ann_ids)}  ({len(ann_ids)/max(len(img_ids),1):.2f} boxes/img)")
    if areas:
        a = np.array(areas)
        print(f"  box area px : min {a.min():.1f}  median {np.median(a):.1f}  max {a.max():.1f}")
        print(f"  area (norm) : median {np.median(a)/(IMG_W*IMG_H):.6f}   (project mean ~0.000364)")
        print(f"  median side : {np.sqrt(np.median(a)):.1f} px")


loading annotations into memory...
Done (t=7.45s)
creating index...
index created!

[train]  instances_train.json
  categories  : ['uav'] (id [1])
  images      : 122488  (background-only: 138)
  annotations : 2564104  (20.93 boxes/img)
  box area px : min 0.9  median 65.4  max 4344.6
  area (norm) : median 0.000200   (project mean ~0.000364)
  median side : 8.1 px
loading annotations into memory...
Done (t=2.06s)
creating index...
index created!

[val]  instances_val.json
  categories  : ['uav'] (id [1])
  images      : 30093  (background-only: 1)
  annotations : 604854  (20.10 boxes/img)
  box area px : min 4.2  median 61.5  max 3128.7
  area (norm) : median 0.000188   (project mean ~0.000364)
  median side : 7.8 px


In [7]:
%%writefile /kaggle/working/yolox_uav_s.py
from yolox.exp import Exp as MyExp


class Exp(MyExp):
    def __init__(self):
        super().__init__()

        # ---- Model: YOLOX-S (capacity parity with YOLOv8s) ----
        self.depth = 0.33
        self.width = 0.50
        self.num_classes = 1                       # single class: uav
        self.exp_name = "yolox_uav_s"

        # ---- Data ----
        # COCODataset resolves images at data_dir/<train2017|val2017>/<file_name>
        # and annotations at data_dir/annotations/<*_ann>. Cell 4 builds exactly that.
        self.data_dir = "/kaggle/working/yolox_data"
        self.train_ann = "instances_train.json"
        self.val_ann = "instances_val.json"

        # ---- Input: NATIVE 640x512, (h, w); both divisible by stride 32 ----
        self.input_size = (512, 640)
        self.test_size = (512, 640)
        self.multiscale_range = 0                  # fix size; multi-scale jitters DOWN too, which hurts tiny objects

        # ---- Augmentation: throttled for tiny + dense objects ----
        self.mosaic_prob = 1.0
        self.mosaic_scale = (0.8, 1.2)             # default (0.5,1.5) shrinks 8px boxes below resolvability
        self.enable_mixup = False                  # blends faint tiny targets into ambiguous labels
        self.mixup_prob = 0.0
        self.degrees = 0.0                         # rotation interpolates tiny boxes away
        self.shear = 0.0
        self.translate = 0.1
        self.hsv_prob = 1.0                        # near-no-op on thermal IR, harmless
        self.flip_prob = 0.5                       # horizontal flip is safe for this scene

        # ---- Schedule: matched protocol with YOLOv8s ----
        self.max_epoch = 50
        self.no_aug_epochs = 10                    # last 10 epochs: aug off (scaled from YOLOX's 15-of-300)
        self.warmup_epochs = 5
        self.basic_lr_per_img = 0.01 / 64.0        # YOLOX's tuned convention -> 0.0025 at batch 16
        self.scheduler = "yoloxwarmcos"
        self.weight_decay = 5e-4                   # same as YOLOv8s
        self.momentum = 0.9                        # YOLOX-tuned (vs YOLOv8s 0.937; each framework-calibrated)
        self.ema = True

        # ---- Eval / logging ----
        self.eval_interval = 1                     # evaluate EVERY epoch — parity with YOLOv8s
        self.print_interval = 50
        self.data_num_workers = 4                  # drop to 2 if loader is CPU/RAM-bound on Kaggle
        self.output_dir = "/kaggle/working/YOLOX_outputs"

        # NOTE: this YOLOX version builds train/val datasets *inline* and caps labels at
        # 120 boxes per (mosaic) sample. Given ~21 boxes/frame, the densest mosaics may
        # clip a few boxes. If that proves to matter we'll raise the cap before the full
        # run; left at the default for now. (Overriding get_dataset() does NOT change
        # this in 0.3.0 — get_data_loader builds the dataset directly.)


Writing /kaggle/working/yolox_uav_s.py


In [10]:
import shutil, torch
from pathlib import Path

CKPT_DATASET = Path("/kaggle/input/uav-yolox-checkpoints")     # attach via "Add Data" from session 2 on
OUT_DIR = Path("/kaggle/working/YOLOX_outputs/yolox_uav_s")
OUT_DIR.mkdir(parents=True, exist_ok=True)
LATEST = OUT_DIR / "latest_ckpt.pth"

def resumable(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)   # our own trusted ckpt
    return {"model", "optimizer", "start_epoch"} <= set(ck.keys()), ck.get("start_epoch"), set(ck.keys())

if LATEST.exists():
    RESUME = True
    print("latest_ckpt.pth already in working dir — will resume.")
elif (CKPT_DATASET / "latest_ckpt.pth").exists():
    shutil.copy(CKPT_DATASET / "latest_ckpt.pth", LATEST)
    if (CKPT_DATASET / "best_ckpt.pth").exists():
        shutil.copy(CKPT_DATASET / "best_ckpt.pth", OUT_DIR / "best_ckpt.pth")  # keep best-AP tracking continuous
    ok, start_epoch, keys = resumable(LATEST)
    if not ok:
        raise RuntimeError(f"Restored latest_ckpt.pth is NOT resumable — keys: {sorted(keys)}. "
                           "Refusing to proceed (resuming on this would silently restart from scratch).")
    RESUME = True
    print(f"Restored resumable checkpoint — resumes at epoch {start_epoch} (optimizer + start_epoch present).")
else:
    RESUME = False
    print("No prior checkpoint anywhere -> FRESH start from COCO-pretrained yolox_s.pth.")

print(f"\nRESUME = {RESUME}")


latest_ckpt.pth already in working dir — will resume.

RESUME = True


In [11]:
import subprocess, signal, time, os
from pathlib import Path

TRAIN_PY = "/kaggle/working/YOLOX/tools/train.py"
EXP_FILE = "/kaggle/working/yolox_uav_s.py"
OUT_DIR  = Path("/kaggle/working/YOLOX_outputs/yolox_uav_s")
LATEST, LOG = OUT_DIR / "latest_ckpt.pth", OUT_DIR / "train_log.txt"
TIME_BUDGET_SEC = int(9.0 * 3600)

base = ["python", "-u", TRAIN_PY, "-f", EXP_FILE, "-d", "1", "-b", "16", "--fp16"]  # -u = unbuffered
cmd  = base + (["--resume"] if RESUME else ["-c", "/kaggle/working/yolox_s.pth"])
print("Launching:", " ".join(cmd), "\n", flush=True)

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
t0 = time.time()
proc = subprocess.Popen(cmd, cwd="/kaggle/working/YOLOX", env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

watchdog = False
try:
    for line in proc.stdout:                       # stream live
        print(line, end="", flush=True)
        if time.time() - t0 > TIME_BUDGET_SEC:
            watchdog = True
            print(f"\n[watchdog] {TIME_BUDGET_SEC/3600:.1f}h reached — stopping for upload.", flush=True)
            proc.send_signal(signal.SIGINT); break
    proc.wait(timeout=180)
except KeyboardInterrupt:
    print("\n[interrupted] stopping training cleanly...", flush=True)
    proc.send_signal(signal.SIGINT)
    try: proc.wait(timeout=120)
    except subprocess.TimeoutExpired: proc.terminate()
    raise
finally:
    if proc.poll() is None:
        try: proc.terminate(); proc.wait(timeout=60)
        except Exception: proc.kill()

progressed = LATEST.exists() and LATEST.stat().st_mtime >= t0
if watchdog:
    print(f"\nWatchdog-stopped; checkpoint {'updated' if progressed else 'NOT updated — check log!'}.")
elif progressed:
    print("\nTraining completed and latest_ckpt.pth was written.")
else:
    raise RuntimeError("Training exited WITHOUT writing latest_ckpt.pth — it crashed. See output above.")

Launching: python -u /kaggle/working/YOLOX/tools/train.py -f /kaggle/working/yolox_uav_s.py -d 1 -b 16 --fp16 --resume 

2026-06-02 13:41:34.161066: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780407694.191020     250 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780407694.200274     250 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780407694.222232     250 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780407694.222292     250 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [12]:
import json, os, shutil, subprocess, torch
from pathlib import Path
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = s.get_secret("KAGGLE_USERNAME")   # reuse the secrets you set up for YOLOv8s
os.environ["KAGGLE_KEY"]      = s.get_secret("KAGGLE_KEY")

DATASET_ID = "lazarvelinov46/uav-yolox-checkpoints"
OUT_DIR = Path("/kaggle/working/YOLOX_outputs/yolox_uav_s")
STAGING = Path("/kaggle/working/yolox_ckpt_staging")
if STAGING.exists(): shutil.rmtree(STAGING)
STAGING.mkdir(parents=True)

for name in ["latest_ckpt.pth", "best_ckpt.pth", "train_log.txt"]:
    src = OUT_DIR / name
    if src.exists():
        shutil.copy(src, STAGING / name); print(f"Staged {name} ({src.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"Skip {name} (absent)")

# Verify we're persisting a RESUMABLE file — the guard that would have caught the
# YOLOv8s stripped-checkpoint failure at upload time.
epoch = "?"
if (STAGING / "latest_ckpt.pth").exists():
    ck = torch.load(STAGING / "latest_ckpt.pth", map_location="cpu", weights_only=False)
    assert {"optimizer", "start_epoch"} <= set(ck.keys()), "Staged latest_ckpt is NOT resumable — aborting upload."
    epoch = ck.get("start_epoch", "?")

json.dump({"title": "UAV YOLOX Checkpoints", "id": DATASET_ID, "licenses": [{"name": "CC0-1.0"}]},
          open(STAGING / "dataset-metadata.json", "w"), indent=2)

def run(c):
    r = subprocess.run(c, capture_output=True, text=True); print(r.stdout)
    if r.returncode: print("STDERR:", r.stderr)
    return r.returncode

msg = f"end of session, resumes at epoch {epoch}"
print(f"\nVersion message: {msg!r}")
if run(["kaggle","datasets","version","-p",str(STAGING),"-m",msg,"--dir-mode","zip"]) != 0:
    print("Versioning failed — creating dataset for the first time...")
    run(["kaggle","datasets","create","-p",str(STAGING),"--dir-mode","zip"])
print("Upload finished.")


Staged latest_ckpt.pth (71.8 MB)
Staged best_ckpt.pth (71.8 MB)
Staged train_log.txt (0.6 MB)

Version message: 'end of session, resumes at epoch 12'
Starting upload for file train_log.txt
Upload successful: train_log.txt (588KB)
Starting upload for file best_ckpt.pth
Upload successful: best_ckpt.pth (69MB)
Starting upload for file latest_ckpt.pth
Upload successful: latest_ckpt.pth (69MB)
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion

STDERR: 
  0%|          | 0.00/588k [00:00<?, ?B/s]
100%|██████████| 588k/588k [00:00<00:00, 1.21MB/s]

  0%|          | 0.00/68.5M [00:00<?, ?B/s]
 16%|█▌        | 10.6M/68.5M [00:00<00:00, 77.9MB/s]
 42%|████▏     | 29.0M/68.5M [00:00<00:00, 104MB/s] 
 60%|█████▉    | 41.0M/68.5M [00:00<00:00, 112MB/s]
 79%|███████▊  | 53.8M/68.5M [00:00<00:00, 106MB/s]
 99%|█████████▉| 68.1M/68.5M [00:00<00:00, 119MB/s]
100%|██████████| 68.5M/68.5M [00:01<00:00, 61.4MB/s]

  0%|          | 0.00/68.5M [00: